In [3]:
!pip install google_play_scraper
!pip install textblob
from google_play_scraper import app
import pandas as pd
import numpy as np
import sklearn
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as dates
import seaborn as sns
import textblob
#from wordcloud import WordCloud
from pathlib import Path
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix,classification_report, accuracy_score

import pickle
import re
import time
import datetime                              # access to %%time, for timing individual notebook cells
import os
from PIL import Image
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

%matplotlib inline
%config InlineBackend.figure_format='retina'

# plt.style.use('seaborn')
plt.rcParams["figure.figsize"] = (15,10)

In [4]:

from google_play_scraper import app, Sort, reviews_all

nhs_reviews = reviews_all(
    'com.pure.indosat.care',
    sleep_milliseconds=0, # defaults to 0
    lang='en', # defaults to 'en'
    sort=Sort.NEWEST, # defaults to Sort.MOST_RELEVANT
)

In [7]:
#Save the NHS Apps reviews into dataframe
df_nhsrev = pd.DataFrame(np.array(nhs_reviews),columns=['content'])
df_nhsrev = df_nhsrev.join(pd.DataFrame(df_nhsrev.pop('content').tolist()))
df_nhsrev.to_csv(r'C:\Users\alqis\Documents\crqwling.csv', index=False)
df_nhsrev.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,3e286299-63ac-4c2b-b58f-12a28b38f4f3,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,"scam game, greedy fks",1,0,82.13.0,2026-04-06 22:19:40,"Good afternoon, IM3 Customer. We apologize for...",2026-04-07 11:18:26,82.13.0
1,a3780d29-c7cc-4903-b465-6306588e350a,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,"Jadi yg lebih baik, jangan hanya kuantitas tap...",2,0,82.13.0,2026-04-06 21:36:21,Terima kasih ka Boy Angky untuk saran yang dib...,2019-12-02 10:42:51,82.13.0
2,3b01ef8a-7643-4255-a74d-f06832166415,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,"jaringan bapuk, gak tau harus complain kemana,...",1,0,82.13.0,2026-04-06 20:30:16,"Selamat pagi Bapak Supriyanto, mohon maaf sebe...",2026-04-06 08:58:02,82.13.0
3,614d6f63-9655-479e-8068-4a592d460774,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,"ngelunjak, tambah banyak ngaku 1 bulan tapi 28...",1,0,None,2026-04-06 17:55:14,"Selamat pagi Bapak Yayan, mohon maaf untuk yan...",2026-04-07 09:10:49,None
4,26add895-ba87-496e-8b01-ed56001c020d,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,"entah ini dr kartunya atau aplikasinya,tapi ny...",3,0,82.13.0,2026-04-06 11:52:02,None,NaT,82.13.0


In [8]:
# labeling sederhana berdasarkan rating dari pengguna 
def label_sentiment(score):
    if score <= 2:
        return 'Negative'
    elif score == 3:
        return 'Neutral'
    else:
        return 'Positive'

df_nhsrev['sentiment'] = df_nhsrev['score'].apply(label_sentiment)
negatif = df_nhsrev[df_nhsrev['sentiment'] == 'Negative'].sample(n=50)
netral  = df_nhsrev[df_nhsrev['sentiment'] == 'Neutral'].sample(n=50)
positif = df_nhsrev[df_nhsrev['sentiment'] == 'Positive'].sample(n=50)

df_sample = pd.concat([negatif, netral, positif])
df_sample.head()
df_sample.to_csv(r'C:\Users\alqis\Documents\crawling.csv', index=False)